# STREAMSENSE — Quick Voice Test
Record your voice and predict the spoken command.

**Requirements:** `pip install sounddevice scipy numpy torch torchaudio`

**Before running:** Make sure `best_model.pth` is in `C:\STREAMSENSE\checkpoints\`

In [ ]:
# Cell 1 — Imports and paths
import sys
import json
import time
import numpy as np
import torch
import torch.nn as nn
import sounddevice as sd
import scipy.io.wavfile as wav
from pathlib import Path

# Add training folder to path so we can import mel_pipeline and model
STREAMSENSE_ROOT = Path(r'C:\STREAMSENSE')
sys.path.insert(0, str(STREAMSENSE_ROOT / 'training'))

from mel_pipeline import preprocess
from model import StreamSenseNet

# Paths
CKPT_PATH    = STREAMSENSE_ROOT / 'checkpoints' / 'best_model.pth'
LABELS_PATH  = STREAMSENSE_ROOT / 'class_labels.json'
RECORDS_DIR  = STREAMSENSE_ROOT / 'recordings'
RECORDS_DIR.mkdir(exist_ok=True)

# MPIC params
SAMPLE_RATE  = 16000
FRAME_LEN    = 16000    # 1 second
HOP          = 8000     # 50% overlap
CONF_THRESH  = 0.70

print('Imports OK')
print(f'Checkpoint : {CKPT_PATH}')
print(f'Exists     : {CKPT_PATH.exists()}')

In [ ]:
# Cell 2 — Load class labels
with open(LABELS_PATH, 'r') as f:
    raw = json.load(f)

# class_labels.json format: {"0": "yes", "1": "no", ...}
CLASS_LABELS = {int(k): v for k, v in raw.items()}
print('Class labels:')
for idx, label in CLASS_LABELS.items():
    print(f'  {idx} → {label}')

In [ ]:
# Cell 3 — Load model
device = torch.device('cpu')   # local machine, CPU inference

model = StreamSenseNet(num_classes=10)
ckpt  = torch.load(CKPT_PATH, map_location='cpu')
model.load_state_dict(ckpt['model_state'])
model.eval()

print(f'Model loaded from epoch {ckpt["epoch"]}')
print(f'Checkpoint val_acc : {ckpt["val_accuracy"]:.2f}%')

In [ ]:
# Cell 4 — Predict function (sliding window)
# Handles any length recording — slides 1-second windows with 50% overlap

def predict_audio(audio: np.ndarray, sample_rate: int) -> dict:
    """
    Run sliding window prediction over audio of any length.
    
    Args:
        audio       : float32 numpy array, any length, mono
        sample_rate : sample rate of the audio
    
    Returns:
        dict with predicted label, confidence, and per-window results
    """
    # Resample if needed
    if sample_rate != SAMPLE_RATE:
        import torchaudio.functional as F
        import torch
        t = torch.from_numpy(audio).unsqueeze(0)
        t = F.resample(t, sample_rate, SAMPLE_RATE)
        audio = t.squeeze(0).numpy()
        print(f'Resampled {sample_rate}Hz → {SAMPLE_RATE}Hz')

    # Normalize audio amplitude to [-1, 1]
    max_val = np.abs(audio).max()
    if max_val > 0:
        audio = audio / max_val

    n_samples = len(audio)
    print(f'Audio length : {n_samples} samples ({n_samples/SAMPLE_RATE:.2f}s)')

    # Build windows
    windows = []
    if n_samples <= FRAME_LEN:
        # Pad short audio to exactly FRAME_LEN
        padded = np.zeros(FRAME_LEN, dtype=np.float32)
        padded[:n_samples] = audio
        windows.append(padded)
    else:
        # Slide with 50% overlap
        start = 0
        while start + FRAME_LEN <= n_samples:
            windows.append(audio[start : start + FRAME_LEN].astype(np.float32))
            start += HOP
        # Last window — pad if needed
        if start < n_samples:
            last = np.zeros(FRAME_LEN, dtype=np.float32)
            last[:n_samples - start] = audio[start:]
            windows.append(last)

    print(f'Windows      : {len(windows)}')

    # Run inference on each window
    results = []
    softmax = nn.Softmax(dim=1)

    for i, window in enumerate(windows):
        tensor = preprocess(window)              # [1, 1, 64, 97]
        tensor = tensor.squeeze(0)               # [1, 64, 97] — remove batch from pipeline
        tensor = tensor.unsqueeze(0)             # [1, 1, 64, 97] — add batch dim for model

        with torch.no_grad():
            logits = model(tensor)               # [1, 10]
            probs  = softmax(logits)[0]          # [10]

        conf, pred_idx = probs.max(dim=0)
        pred_label     = CLASS_LABELS[pred_idx.item()]
        confidence     = conf.item()

        results.append({
            'window'    : i + 1,
            'label'     : pred_label,
            'confidence': confidence,
            'probs'     : probs.numpy(),
        })

        print(f'  Window {i+1}/{len(windows)} : {pred_label:6s}  confidence={confidence:.3f}')

    # Pick window with highest confidence
    best = max(results, key=lambda r: r['confidence'])

    return {
        'prediction' : best['label'] if best['confidence'] >= CONF_THRESH else 'unclear',
        'confidence' : best['confidence'],
        'threshold'  : CONF_THRESH,
        'windows'    : results,
        'best_window': best['window'],
    }


print('Predict function ready.')

In [ ]:
# Cell 5 — Record your voice
# Change WORD to whatever you want to say
# Adjust DURATION_SEC if you want more recording time

WORD         = 'yes'       # what you will say — just for filename
DURATION_SEC = 2           # seconds to record
REC_SR       = 16000       # record directly at 16kHz — no resampling needed

print(f'Recording {DURATION_SEC}s at {REC_SR}Hz')
print(f'Say "{WORD}" clearly after the beep...')
time.sleep(0.5)
print('Recording NOW...')

recording = sd.rec(
    int(DURATION_SEC * REC_SR),
    samplerate = REC_SR,
    channels   = 1,
    dtype      = 'float32'
)
sd.wait()   # block until recording done

audio = recording.squeeze()   # [N] float32

# Save to recordings/
save_path = RECORDS_DIR / f'my_{WORD}_{int(time.time())}.wav'
wav.write(str(save_path), REC_SR, audio)

print(f'Done. Saved → {save_path}')
print(f'Samples : {len(audio)}')
print(f'Duration: {len(audio)/REC_SR:.2f}s')
print(f'Max amp : {np.abs(audio).max():.4f}')

In [ ]:
# Cell 6 — Run prediction on recorded audio

print('=' * 50)
print('STREAMSENSE — Prediction')
print('=' * 50)

result = predict_audio(audio, REC_SR)

print()
print('─' * 50)
if result['prediction'] == 'unclear':
    print(f'  Result     : UNCLEAR')
    print(f'  Confidence : {result["confidence"]:.3f}  (threshold={CONF_THRESH})')
    print(f'  Tip        : Speak louder or closer to mic')
else:
    print(f'  Prediction : {result["prediction"].upper()}')
    print(f'  Confidence : {result["confidence"]:.3f}')
    print(f'  Best window: {result["best_window"]}')
print('─' * 50)

In [ ]:
# Cell 7 — OPTIONAL: predict from an existing WAV file
# Use this if you recorded on your phone and transferred the file

import torchaudio

WAV_FILE = r'C:\STREAMSENSE\recordings\my_yes_test.wav'  # change this path

waveform, sr = torchaudio.load(WAV_FILE)

# Convert to mono if stereo
if waveform.shape[0] > 1:
    waveform = waveform.mean(dim=0, keepdim=True)

audio_file = waveform.squeeze(0).numpy()   # [N] float32

print('=' * 50)
print(f'File: {Path(WAV_FILE).name}')
print('=' * 50)

result = predict_audio(audio_file, sr)

print()
print('─' * 50)
if result['prediction'] == 'unclear':
    print(f'  Result     : UNCLEAR')
    print(f'  Confidence : {result["confidence"]:.3f}')
else:
    print(f'  Prediction : {result["prediction"].upper()}')
    print(f'  Confidence : {result["confidence"]:.3f}')
print('─' * 50)

In [ ]:
# Cell 8 — Full confidence breakdown for all 10 classes
# Run after Cell 6 to see scores for every class

best_window = result['windows'][result['best_window'] - 1]

print('Confidence scores — all classes:')
print(f'{"Class":<10} {"Score":<10} {"Bar"}')
print('─' * 40)

probs = best_window['probs']
order = np.argsort(probs)[::-1]   # sort descending

for idx in order:
    label = CLASS_LABELS[idx]
    score = probs[idx]
    bar   = '█' * int(score * 30)
    marker = ' ← predicted' if idx == order[0] else ''
    print(f'{label:<10} {score:.4f}    {bar}{marker}')